# DirectionOfTrafficFlowRestriction: GeoJSON → pivoted File Geodatabase

## Purpose

Flatten the `DirectionOfTrafficFlowRestriction` array out of a DDCT / TTOM GeoJSON export into a
**File Geodatabase feature class** with **one feature per input feature** and **one column per
vehicle type**, each holding that vehicle's `ValidityDirection`.

**No `arcpy` is involved.** `arcpy` ships only inside ArcGIS Pro's own bundled Python (Windows,
licence-gated, never on PyPI), so `import arcpy` cannot work in a normal venv. This notebook writes
the `.gdb` through GDAL's `OpenFileGDB` driver via `pyogrio`, which is pip-installable, works on
Windows/macOS/Linux, and — unlike `arcpy.conversion.TableToTable`, which yields a table with no
geometry — produces a real **spatial** feature class.

## Pivot rule

```json
"DirectionOfTrafficFlowRestriction": [
  { "VehicleType": ["Taxi", "PublicBus"], "ValidityDirection": "InPositiveDirection" },
  { "VehicleType": ["PassengerCar", "Resident", "MediumTruck"], "ValidityDirection": "InBothDirections" }
],
"FRC": "OtherMajorRoad",
"Net2Class": "Net2Class2",
"FormOfWay": "DualCarriageway"
```

becomes a single feature — each vehicle type is a column, and its cell is the `ValidityDirection`
of the entry that listed it:

| uuid | coordinates_wkt | Taxi | PublicBus | PassengerCar | Resident | MediumTruck | FRC | Net2Class | FormOfWay |
|---|---|---|---|---|---|---|---|---|---|
| 00004531-...-000003d07565 | LINESTRING (...) | InPositiveDirection | InPositiveDirection | InBothDirections | InBothDirections | InBothDirections | OtherMajorRoad | Net2Class2 | DualCarriageway |

…plus the geometry itself, so the layer draws on a map.

The vehicle-type columns are **discovered from the data** in pass 1, not hardcoded, so a vehicle
that only appears deep in the file still gets a column.

A vehicle in none of a feature's entries gets **NULL** — deliberately distinct from a vehicle that
is present but unrestricted.

### Two data conditions this handles rather than hides

**1. The same vehicle in more than one entry, with different directions.** In the ESP test export
this happens 1,378 times — typically a time-limited restriction plus a default one:

```json
[{ "VehicleType": ["PassengerCar", ...], "ValidityPeriod": "[(h23){h1}]", "ValidityDirection": "InNegativeDirection" },
 { "VehicleType": ["PassengerCar", ...],                                 "ValidityDirection": "InPositiveDirection" }]
```

One cell cannot hold both. Assigning `row[vehicle] = validity` in a loop — the obvious
implementation — keeps only the **last** entry and silently discards the other, losing a real
restriction. Instead the values are joined in entry order:
`InNegativeDirection|InPositiveDirection`, and the run reports how many cells ended up
multi-valued so the number is never a surprise.

**2. `ValidityPeriod`.** 297 features in the ESP export carry it, and the pivoted layout has
nowhere to put it. By default those values are **not in the output** and the run prints how many
were omitted. Set `include_secondary_fields = True` to add a `<Vehicle>_ValidityPeriod` column per
vehicle; its pipe-separated values line up position-by-position with that vehicle's direction
cell, so `InNegativeDirection|InPositiveDirection` pairs with `[(h23){h1}]|`.

There is deliberately no standalone `ValidityDirection` column: every direction value now lives in
a vehicle column, so a single shared one would have nothing to hold.

## Built for a ~5 GB input on a 16 GB machine

A 5 GB GeoJSON cannot be `json.load`-ed — parsed into Python objects it needs several times its
file size in RAM. Nothing here holds the whole dataset:

* **`ijson` streams the features** one at a time off disk (`yajl2_c` C backend).
* **The layer is written in batches** — `batch_size` features are accumulated, appended to the
  `.gdb`, then dropped. Peak memory is one batch, a few hundred MB, regardless of input size.
* **Summary statistics are streaming counters**, not `value_counts()` over a full DataFrame.

The file is read **twice**: pass 1 discovers the column names (a `.gdb` layer's schema is fixed
when the first feature is written), pass 2 writes. Pass 1 only parses `properties` and skips the
coordinate arrays, so it is much the cheaper of the two.

## Progress bars

Both passes show a `tqdm` bar measuring **bytes consumed from the file**, not features processed:
the feature count is unknown until the file has been read once, whereas the file size is known up
front, so the percentage, throughput and ETA are meaningful from the first refresh. Each bar's
postfix carries the running feature and row counts. Bars refresh every
`progress_refresh_features` features, since each refresh costs a `tell()` plus a redraw. If `tqdm`
is missing the notebook still runs — a no-op stand-in takes over.

## Output

`<prefix>_dtfr.gdb` — nothing else, and no other file format. A `.gdb` is a **directory**, not a
file: copy or zip the whole folder when sharing it.

In [1]:
# Install the packages needed to stream the JSON and write a File Geodatabase.
# ijson              -> incremental JSON parser, so the 5 GB input is never fully materialised.
# pyogrio + geopandas -> GDAL bindings; the wheels bundle GDAL, so there is no system gdal-config
#                       step and NO ArcGIS / arcpy / FileGDB SDK requirement.
#                       (Do NOT use fiona: it builds from source and fails with
#                       "A GDAL API version must be specified".)
# shapely            -> builds geometry from the GeoJSON coordinates.
# tqdm               -> the progress bars for both passes.
# pandas             -> batch assembly and the small preview table.
# %pip installs into the notebook's Python env; the kernel restarts automatically afterwards.
%pip install ijson geopandas pyogrio shapely pyproj tqdm pandas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---------------------------------------------------------------------------
# Configuration — set the input GeoJSON and where the .gdb should land.
# On Databricks, prefix DBFS paths with /dbfs so the local FUSE mount is used.
# ---------------------------------------------------------------------------
import json
import os
import re
import shutil
import time
from collections import Counter

import ijson

# The progress bars measure BYTES READ (see the notes at the top). tqdm.auto picks the notebook
# widget when ipywidgets is available and falls back to a text bar otherwise.
try:
    from tqdm.auto import tqdm
except ImportError:  # keep the notebook runnable without tqdm installed
    print("tqdm not installed — running without progress bars (%pip install tqdm to get them)")

    class tqdm:  # minimal stand-in supporting the calls used below
        def __init__(self, **kwargs):
            self.n = 0

        def __enter__(self):
            return self

        def __exit__(self, *exc):
            return False

        def update(self, n=1):
            self.n += n

        def set_postfix_str(self, *args, **kwargs):
            pass

# >>> Set the input JSON / GeoJSON path here <<<
input_json_path = (
    "/Users/sawan.darekar/Desktop/workspace_data/external-work/"
    "ESP_DTER_test_GeoJSON_00005858-5800-1200-0000-00007d2ca82a_202607221727.json"
)

# >>> Set the output directory and file prefix here <<<
output_dir = os.path.dirname(input_json_path)
output_prefix = os.path.splitext(os.path.basename(input_json_path))[0]

gdb_output_path = os.path.join(output_dir, f"{output_prefix}_dtfr.gdb")
gdb_layer_name = "dtfr"

# The nested array to pivot: each value of `pivot_key_field` becomes a column, and the cell holds
# `pivot_value_field` from the entry that listed it. Change these to reuse the notebook for another
# restriction array.
array_field = "DirectionOfTrafficFlowRestriction"
pivot_key_field = "VehicleType"
pivot_value_field = "ValidityDirection"

# Feature-level properties (FRC / Net2Class / FormOfWay / ...) are appended after the vehicle
# columns. They are discovered from the data; these two are excluded because they are constant for
# a whole export and only add noise — drop them from the set to keep them.
excluded_property_fields = {"apiType", "ddctType"}

# Content toggles
include_features_without_restriction = False  # True -> also write features whose restriction array
#                                               is absent, with every vehicle column NULL
include_secondary_fields = False  # True -> add a <Vehicle>_<field> column for every restriction
#                                   field other than the pivoted one (here: ValidityPeriod).
#                                   False keeps the schema minimal but omits those values; the
#                                   write pass reports how many were left out either way.
include_wkt_column = True  # keep coordinates_wkt as a text attribute. The geometry already carries
#                            the shape, so setting False costs no information and cuts the output
#                            size by roughly a third.
multi_value_separator = "|"  # joins the values when one vehicle appears in several entries

# ---------------------------------------------------------------------------
# COORDINATE PRECISION — why this matters and what to set it to.
#
# A File Geodatabase does NOT store coordinates as doubles. Every coordinate is stored as an
# integer on a fixed grid: round((coord - origin) * XYSCALE). Anything finer than 1/XYSCALE is
# snapped away permanently, which is exactly how coordinates come out looking truncated.
#
# GDAL's default XYSCALE is 1e9 -> a 1e-9 degree grid (about 0.1 mm). Source coordinates with up to
# 9 decimal places survive that exactly; a 10th decimal or beyond does not. TTOM/DDCT exports carry
# 7 decimals (~1 cm), so 1e9 is already lossless for them — but the default is raised here so the
# question does not arise for a finer source, and the audit cell at the end MEASURES the real
# deviation on your data rather than assuming.
#
# The grid must also cover the coordinate range within a signed 64-bit integer:
# max |coord| * XYSCALE < 9.2e18. At 1e12 and lon/lat degrees that is 4e14 — a wide margin.
# 1e12 is a ~0.1 micrometre grid, far finer than double precision can express at these magnitudes,
# so nothing measurable is lost. Lower it to 1e9 if a downstream ArcGIS workflow expects the
# standard WGS84 resolution.
# ---------------------------------------------------------------------------
gdb_xy_scale = 1e12
precision_audit_features = 2_000  # features whose source coordinates are kept in memory so the
#                                   audit cell can compare them against what the .gdb stored
max_geometry_deviation_mm = 1.0  # the audit fails if a stored coordinate sits further than
#                                  this from its source. Guards against a grid so coarse that
#                                  the snapping IS the truncation — checking the grid step alone
#                                  would call that 'expected' and pass.

# Streaming / memory knobs — these are what keep a 5 GB input inside 16 GB of RAM
batch_size = 50_000  # features buffered before being appended to the .gdb and dropped
progress_refresh_features = 2_000  # features between progress-bar refreshes
schema_scan_limit = 0  # pass 1: 0 = scan the whole file (exact). A positive number stops early
#                        and is faster, but a vehicle type or property first used late in the file
#                        then gets no column. Values for unknown vehicles are counted and reported
#                        rather than silently dropped.
counter_max_distinct = 200  # cap per-column distinct values tracked, so the counters stay bounded

input_size_bytes = os.path.getsize(input_json_path)
os.makedirs(output_dir, exist_ok=True)

# Rough output estimate. Measured on the ESP exports: 0.49-0.53x the input with the WKT column and
# 0.37x without; the ratios below are rounded up so the warning errs toward caution. Disk, not RAM,
# is the limit that actually bites on a 5 GB input.
size_ratio = 0.6 if include_wkt_column else 0.45
estimated_output_bytes = input_size_bytes * size_ratio
free_bytes = shutil.disk_usage(output_dir).free

print(f"Input        : {input_json_path}")
print(f"Input size   : {input_size_bytes / 1024**3:.2f} GiB")
print(f"GDB out      : {gdb_output_path} (layer: {gdb_layer_name})")
print(f"Est. output  : ~{estimated_output_bytes / 1024**3:.1f} GiB (rough, ~{size_ratio}x input)")
print(f"Free on disk : {free_bytes / 1024**3:.1f} GiB")
print(f"ijson backend: {ijson.backend}")
print(
    f"XY grid      : 1/{gdb_xy_scale:.0e} = {1 / gdb_xy_scale:.1e} deg "
    f"(~{1 / gdb_xy_scale * 111_320_000:.3g} mm at the equator)"
)

if free_bytes < estimated_output_bytes * 1.5:
    print(
        "\n  WARNING: free disk space is close to the estimated output size. Set\n"
        "  include_wkt_column = False (the geometry still carries the shape), or point output_dir\n"
        "  at a bigger volume, before running the write pass."
    )

if ijson.backend == "python":
    print(
        "\n  WARNING: the pure-Python ijson backend is in use and is roughly an order of magnitude\n"
        "  slower — a 5 GB file would take hours. Reinstall ijson so the yajl2_c wheel is used."
    )

Input        : /Users/sawan.darekar/Desktop/workspace_data/external-work/ESP_DTER_test_GeoJSON_00005858-5800-1200-0000-00007d2ca82a_202607221727.json
Input size   : 0.02 GiB
GDB out      : /Users/sawan.darekar/Desktop/workspace_data/external-work/ESP_DTER_test_GeoJSON_00005858-5800-1200-0000-00007d2ca82a_202607221727_dtfr.gdb (layer: dtfr)
Est. output  : ~0.0 GiB (rough, ~0.6x input)
Free on disk : 679.5 GiB
ijson backend: yajl2_c
XY grid      : 1/1e+12 = 1.0e-12 deg (~0.000111 mm at the equator)


/Users/sawan.darekar/Desktop/tomtom_git_workspace/databricks-workspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---------------------------------------------------------------------------
# Pass 1 — discover the columns by streaming the file.
#
# A .gdb layer's schema is fixed when the first feature is written, and the vehicle-type columns are
# data-driven, so the file is scanned once for names only. Requesting 'features.item.properties'
# means ijson never builds the coordinate arrays — this pass is far cheaper than the write pass.
# ---------------------------------------------------------------------------
vehicle_values = []  # distinct VehicleType values, in order of first appearance
secondary_fields = []  # restriction-entry fields other than the pivot key/value (ValidityPeriod)
property_fields = []  # feature-level properties (FRC / Net2Class / FormOfWay / ...)
features_scanned = 0
features_with_array = 0

scan_started = time.perf_counter()

with open(input_json_path, "rb") as fh, tqdm(
    total=input_size_bytes,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Pass 1/2 scanning schema",
    smoothing=0.05,
) as bar:
    for properties in ijson.items(fh, "features.item.properties", use_float=True):
        features_scanned += 1

        for key in properties:
            if key in ("uuid", array_field) or key in excluded_property_fields:
                continue
            if key not in property_fields:
                property_fields.append(key)

        entries = properties.get(array_field) or []
        if entries:
            features_with_array += 1

        for entry in entries:
            for key in entry:
                if key in (pivot_key_field, pivot_value_field):
                    continue
                if key not in secondary_fields:
                    secondary_fields.append(key)

            for vehicle in entry.get(pivot_key_field) or []:
                if vehicle not in vehicle_values:
                    vehicle_values.append(vehicle)

        if features_scanned % progress_refresh_features == 0:
            bar.update(fh.tell() - bar.n)  # advance to the current read position
            bar.set_postfix_str(f"{features_scanned:,} features, {len(vehicle_values)} vehicles")

        if schema_scan_limit and features_scanned >= schema_scan_limit:
            bar.set_postfix_str(f"stopped early at schema_scan_limit={schema_scan_limit:,}")
            break

    bar.update(max(0, fh.tell() - bar.n))  # top the bar up to 100% on a clean finish

scan_elapsed = time.perf_counter() - scan_started

Pass 1/2 scanning schema: 100%|██████████| 17.0M/17.0M [00:00<00:00, 121MB/s, 36,000 features, 5 vehicles]


In [4]:
# ---------------------------------------------------------------------------
# Turn the discovered names into a .gdb schema.
#
# File Geodatabase field names are stricter than plain column names: letters, digits and
# underscores only, no leading digit, 64 characters max, and a handful of reserved names
#   (OBJECTID, SHAPE, ...).
# GDAL would quietly "launder" anything invalid; doing it here instead means the mapping is
# explicit and printed, so a renamed column is never a surprise when the layer is opened.
# ---------------------------------------------------------------------------
GDB_RESERVED_NAMES = {"objectid", "shape", "shape_length", "shape_area", "fid", "geometry"}


def gdb_field_name(source_name, used_lower):
    """Return a valid, unique File Geodatabase field name for `source_name`."""
    clean = re.sub(r"[^0-9A-Za-z_]", "_", source_name)
    if not clean or clean[0].isdigit():
        clean = f"f_{clean}"
    clean = clean[:64]
    if clean.lower() in GDB_RESERVED_NAMES:
        clean = f"{clean}_1"

    candidate, suffix = clean, 1
    while candidate.lower() in used_lower:
        suffix += 1
        candidate = f"{clean[:60]}_{suffix}"
    used_lower.add(candidate.lower())
    return candidate


# Logical column order: uuid, optional WKT text, one column per vehicle, optional per-vehicle
# secondary fields, then the feature-level properties.
source_columns = ["uuid"]
if include_wkt_column:
    source_columns.append("coordinates_wkt")
source_columns += list(vehicle_values)
if include_secondary_fields:
    source_columns += [f"{v}_{f}" for v in vehicle_values for f in secondary_fields]
source_columns += property_fields

used_lower = set()
field_name_map = {name: gdb_field_name(name, used_lower) for name in source_columns}
gdb_column_order = [field_name_map[name] for name in source_columns]
renamed_fields = {src: dst for src, dst in field_name_map.items() if src != dst}

# Which .gdb field holds each vehicle's direction, and each (vehicle, field) secondary value.
vehicle_field = {v: field_name_map[v] for v in vehicle_values}
secondary_field_name = (
    {(v, f): field_name_map[f"{v}_{f}"] for v in vehicle_values for f in secondary_fields}
    if include_secondary_fields
    else {}
)
property_field = {p: field_name_map[p] for p in property_fields}
vehicle_value_set = set(vehicle_values)

print(f"Pass 1 took                    : {scan_elapsed:.1f}s")
print(f"Features scanned               : {features_scanned:,}")
print(f"Features with {array_field}: {features_with_array:,}")
print(f"Vehicle types discovered       : {vehicle_values}")
print(f"Property columns discovered    : {property_fields}")
print(f"Property columns excluded      : {sorted(excluded_property_fields)}")
print(f"Other restriction fields found : {secondary_fields}")
print()
print(f"GDB layer schema ({len(gdb_column_order)} fields + geometry):")
print(f"  {', '.join(gdb_column_order)}")

if renamed_fields:
    print("\n  Renamed to satisfy File Geodatabase field-name rules:")
    for src, dst in renamed_fields.items():
        print(f"    {src} -> {dst}")

if secondary_fields and not include_secondary_fields:
    print(
        f"\n  NOTE: {secondary_fields} exist in the restriction entries but are not in the schema.\n"
        f"  Set include_secondary_fields = True to add <Vehicle>_<field> columns. The write pass\n"
        f"  below counts exactly how many values this omits."
    )

Pass 1 took                    : 0.2s
Features scanned               : 36,545
Features with DirectionOfTrafficFlowRestriction: 26,542
Vehicle types discovered       : ['PassengerCar', 'Resident', 'Taxi', 'PublicBus', 'MediumTruck']
Property columns discovered    : ['FRC', 'Net2Class', 'FormOfWay']
Property columns excluded      : ['apiType', 'ddctType']
Other restriction fields found : ['ValidityPeriod']

GDB layer schema (10 fields + geometry):
  uuid, coordinates_wkt, PassengerCar, Resident, Taxi, PublicBus, MediumTruck, FRC, Net2Class, FormOfWay

  NOTE: ['ValidityPeriod'] exist in the restriction entries but are not in the schema.
  Set include_secondary_fields = True to add <Vehicle>_<field> columns. The write pass
  below counts exactly how many values this omits.


In [5]:
# ---------------------------------------------------------------------------
# Row builder and batch assembly.
# ---------------------------------------------------------------------------
import geopandas as gpd
import pandas as pd
import pyogrio
from shapely.geometry import shape

# A File Geodatabase feature class holds ONE geometry family. GDAL fixes it when the first batch
# creates the layer, so a later feature of a different family would fail the append — potentially
# minutes into a long run. The write pass uses this map to detect that case and report it instead.
# Single- and multi-part variants share a family, which is why LineString and MultiLineString can
# live in the same layer.
GEOMETRY_FAMILY = {
    "Point": "point",
    "MultiPoint": "point",
    "LineString": "line",
    "MultiLineString": "line",
    "Polygon": "polygon",
    "MultiPolygon": "polygon",
}


def cell_value(value):
    """Render a value for a .gdb text cell; "" means NULL once the batch is assembled."""
    if value is None:
        return ""
    if isinstance(value, list):
        return multi_value_separator.join(str(v) for v in value)
    if isinstance(value, dict):
        return json.dumps(value, separators=(",", ":"))
    return str(value)


def row_for_feature(feature, stats):
    """Pivot one GeoJSON feature into (values, geometry), or None if it should be skipped.

    `values` is a dict keyed by .gdb field name. `stats` accumulates the conditions worth
    reporting: multi-valued cells, omitted secondary values, and values for vehicle types pass 1
    never saw.
    """
    properties = feature.get("properties") or {}
    geometry_json = feature.get("geometry")

    if not geometry_json or not geometry_json.get("coordinates"):
        stats["skipped_empty_geometry"] += 1
        return None

    entries = properties.get(array_field) or []
    if not entries and not include_features_without_restriction:
        stats["skipped_no_restriction"] += 1
        return None

    # vehicle -> list of directions, one per entry that listed it (entry order preserved, so the
    # secondary values below stay aligned with it position by position).
    directions = {}
    secondary_values = {}

    for entry in entries:
        direction = cell_value(entry.get(pivot_value_field))
        vehicles = entry.get(pivot_key_field) or []
        if not isinstance(vehicles, list):
            vehicles = [vehicles]

        for vehicle in vehicles:
            if vehicle not in vehicle_value_set:
                # Only reachable when schema_scan_limit cut pass 1 short: there is no field for
                # this vehicle, so the value cannot be written. Counted, never silent.
                stats["values_for_unknown_vehicle"] += 1
                stats[f"unknown_vehicle:{vehicle}"] += 1
                continue

            directions.setdefault(vehicle, []).append(direction)

            for field in secondary_fields:
                value = cell_value(entry.get(field))
                if include_secondary_fields:
                    secondary_values.setdefault((vehicle, field), []).append(value)
                elif value:
                    stats["omitted_secondary_values"] += 1

    geometry = shape(geometry_json)
    values = {field_name_map["uuid"]: properties.get("uuid") or ""}
    if include_wkt_column:
        values[field_name_map["coordinates_wkt"]] = geometry.wkt

    for vehicle in vehicle_values:
        found = directions.get(vehicle)
        if not found:
            values[vehicle_field[vehicle]] = ""  # vehicle in no entry -> NULL
            continue
        if len(found) > 1:
            # Same vehicle restricted by several entries. Joining keeps both; overwriting would
            # lose a real restriction. Reported at the end of the write pass.
            stats["multi_valued_cells"] += 1
            if len(set(found)) > 1:
                stats["multi_valued_cells_conflicting"] += 1
        values[vehicle_field[vehicle]] = multi_value_separator.join(found)

    for (vehicle, field), name in secondary_field_name.items():
        values[name] = multi_value_separator.join(secondary_values.get((vehicle, field), []))

    for prop, name in property_field.items():
        values[name] = cell_value(properties.get(prop))

    return values, geometry


def build_batch(rows):
    """Turn a batch of (values, geometry) pairs into a GeoDataFrame with a fixed schema.

    The dtypes are pinned deliberately: appending to an existing .gdb layer requires every batch to
    present the same field types, and a batch where some column happens to be entirely empty would
    otherwise be inferred as float64 and clash with the text field the first batch created.

    Empty strings become None so they land as real NULLs rather than zero-length text. The mask
    tests `!= ""` and `notna()` explicitly rather than going through `astype(bool)`, because
    `bool(nan)` is True — a missing key would otherwise survive as a float in a text field.
    """
    frame = pd.DataFrame([r[0] for r in rows], columns=gdb_column_order)
    for column in gdb_column_order:
        frame[column] = frame[column].astype(object).where(
            frame[column].notna() & frame[column].ne(""), None
        )
    return gpd.GeoDataFrame(frame, geometry=[r[1] for r in rows], crs="EPSG:4326")

In [6]:
# ---------------------------------------------------------------------------
# Pass 2 — stream the features and append them to the .gdb batch by batch.
#
# Nothing accumulates across batches: after each write the rows are dropped, so memory stays flat
# whether the input is 17 MB or 5 GB. The only growing structures are the bounded value counters.
# ---------------------------------------------------------------------------
if "OpenFileGDB" not in pyogrio.list_drivers(write=True):
    raise RuntimeError(
        f"GDAL {pyogrio.__gdal_version_string__} cannot write .gdb — the OpenFileGDB write driver "
        "needs GDAL >= 3.6. Upgrade pyogrio (pip install -U pyogrio)."
    )

# A .gdb is a directory, so a re-run has to remove the whole tree — otherwise the features below
# would be appended to the previous run's layer.
if os.path.isdir(gdb_output_path):
    shutil.rmtree(gdb_output_path)
    print(f"Removed previous {gdb_output_path}")

features_read = 0
rows_written = 0
batches_written = 0
stats = Counter()
# Built with **-unpacking rather than the dict | operator, which needs Python 3.9+ (older
# Databricks runtimes ship 3.8).
value_counters = {
    **{vehicle_field[v]: Counter() for v in vehicle_values},
    **{property_field[p]: Counter() for p in property_fields},
}
overflowed_counters = set()
preview_rows = []
layer_geometry_family = None  # fixed by the first feature written; see GEOMETRY_FAMILY above
audit_source_coords = []  # source coordinates of the first N written features, for the audit cell

batch = []
write_started = time.perf_counter()


def flush(batch):
    """Append one batch to the .gdb; the first write creates the layer, later ones append.

    The XY grid is a layer creation option, so it is only passed on the call that creates the
    layer; later appends inherit it (verified by the audit cell at the end).
    """
    global rows_written, batches_written
    if not batch:
        return
    creating = batches_written == 0
    pyogrio.write_dataframe(
        build_batch(batch),
        gdb_output_path,
        layer=gdb_layer_name,
        driver="OpenFileGDB",
        append=not creating,
        layer_options={"XYSCALE": f"{gdb_xy_scale:.0f}"} if creating else None,
    )
    rows_written += len(batch)
    batches_written += 1


with open(input_json_path, "rb") as fh, tqdm(
    total=input_size_bytes,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Pass 2/2 writing GDB    ",
    smoothing=0.05,
) as bar:
    for feature in ijson.items(fh, "features.item", use_float=True):
        features_read += 1
        raw_coordinates = (feature.get("geometry") or {}).get("coordinates")
        row = row_for_feature(feature, stats)

        if row is not None:
            # One geometry family per feature class. Rather than letting GDAL abort the append
            # (possibly minutes into the run), an incompatible feature is skipped and counted, and
            # the summary below says so loudly.
            geometry_type = row[1].geom_type
            family = GEOMETRY_FAMILY.get(geometry_type, geometry_type)
            if layer_geometry_family is None:
                layer_geometry_family = family
            if family != layer_geometry_family:
                stats["skipped_geometry_type_mismatch"] += 1
                stats[f"mismatched_geometry:{geometry_type}"] += 1
                row = None

        if row is not None:
            # Bounded streaming statistics — a stand-in for value_counts() on a full DataFrame.
            for column, counter in value_counters.items():
                value = row[0].get(column)
                if value in counter or len(counter) < counter_max_distinct:
                    counter[value] += 1
                else:
                    overflowed_counters.add(column)

            if len(preview_rows) < 10:
                preview_rows.append(row[0])

            # Keep the untouched source coordinates of the first N written features so the audit
            # cell can prove what the .gdb did or did not change. Bounded, so memory stays flat.
            if len(audit_source_coords) < precision_audit_features:
                audit_source_coords.append(raw_coordinates)

            batch.append(row)
            if len(batch) >= batch_size:
                flush(batch)
                batch = []

        if features_read % progress_refresh_features == 0:
            bar.update(fh.tell() - bar.n)  # advance to the current read position
            bar.set_postfix_str(
                f"{features_read:,} features -> {rows_written + len(batch):,} rows"
            )

    bar.update(max(0, fh.tell() - bar.n))  # top the bar up to 100%
    bar.set_postfix_str(f"{features_read:,} features -> {rows_written + len(batch):,} rows")

flush(batch)
batch = []
write_elapsed = time.perf_counter() - write_started

# Nothing written means no layer was ever created, so every cell below would fail on a missing
# dataset. Fail here instead, where the cause can be named.
if rows_written == 0:
    raise RuntimeError(
        f"No features written, so no .gdb was created. {features_read:,} features were read; "
        f"{stats['skipped_no_restriction']:,} had no '{array_field}' array and "
        f"{stats['skipped_empty_geometry']:,} had no geometry. If the input genuinely has no "
        f"restrictions, set include_features_without_restriction = True to write the features "
        f"anyway."
    )

Removed previous /Users/sawan.darekar/Desktop/workspace_data/external-work/ESP_DTER_test_GeoJSON_00005858-5800-1200-0000-00007d2ca82a_202607221727_dtfr.gdb


Pass 2/2 writing GDB    : 100%|██████████| 17.0M/17.0M [00:00<00:00, 38.1MB/s, 36,545 features -> 26,542 rows]


In [7]:
print(f"Pass 2 took                    : {write_elapsed / 60:.1f}min")
print(f"Features read                  : {features_read:,}")
print(f"Features written               : {rows_written:,}")
print(f"Batches written                : {batches_written:,} (batch_size={batch_size:,})")
print(f"Skipped, no restriction array  : {stats['skipped_no_restriction']:,}")
print(f"Skipped, empty geometry        : {stats['skipped_empty_geometry']:,}")
print(f"Geometry family of the layer   : {layer_geometry_family}")
print(f"Output                         : {gdb_output_path} (layer: {gdb_layer_name})")
print()

# The data conditions worth knowing about, reported rather than hidden.
if stats["multi_valued_cells"]:
    print(
        f"Multi-valued cells             : {stats['multi_valued_cells']:,} "
        f"(of which {stats['multi_valued_cells_conflicting']:,} hold genuinely different "
        f"directions)\n"
        f"  A vehicle appeared in several restriction entries for the same feature, so its cell\n"
        f"  holds every value joined by '{multi_value_separator}' in entry order."
    )

if stats["omitted_secondary_values"]:
    print(
        f"Omitted {secondary_fields} values : {stats['omitted_secondary_values']:,}\n"
        f"  These are NOT in the .gdb. Set include_secondary_fields = True and re-run to add\n"
        f"  <Vehicle>_<field> columns for them."
    )

if stats["skipped_geometry_type_mismatch"]:
    mismatched = {
        k.split(":", 1)[1]: v for k, v in stats.items() if k.startswith("mismatched_geometry:")
    }
    print(
        f"SKIPPED, geometry family clash : {stats['skipped_geometry_type_mismatch']:,} "
        f"{mismatched}\n"
        f"  A File Geodatabase feature class holds one geometry family, and this layer is\n"
        f"  '{layer_geometry_family}'. These features are NOT in the output — split the input by\n"
        f"  geometry type and run once per type to keep them."
    )

if stats["values_for_unknown_vehicle"]:
    unknown = {k.split(":", 1)[1]: v for k, v in stats.items() if k.startswith("unknown_vehicle:")}
    print(
        f"DROPPED values, no field       : {stats['values_for_unknown_vehicle']:,} {unknown}\n"
        f"  These vehicle types were not seen during pass 1 because schema_scan_limit stopped it\n"
        f"  early. Set schema_scan_limit = 0 and re-run for a complete schema."
    )

Pass 2 took                    : 0.0min
Features read                  : 36,545
Features written               : 26,542
Batches written                : 1 (batch_size=50,000)
Skipped, no restriction array  : 10,003
Skipped, empty geometry        : 0
Geometry family of the layer   : line
Output                         : /Users/sawan.darekar/Desktop/workspace_data/external-work/ESP_DTER_test_GeoJSON_00005858-5800-1200-0000-00007d2ca82a_202607221727_dtfr.gdb (layer: dtfr)

Multi-valued cells             : 1,378 (of which 1,378 hold genuinely different directions)
  A vehicle appeared in several restriction entries for the same feature, so its cell
  holds every value joined by '|' in entry order.
Omitted ['ValidityPeriod'] values : 1,272
  These are NOT in the .gdb. Set include_secondary_fields = True and re-run to add
  <Vehicle>_<field> columns for them.


In [8]:
# First rows, as a sanity check on the pivot. Only these 10 were kept in memory.
preview = pd.DataFrame(preview_rows, columns=gdb_column_order)
if include_wkt_column and not preview.empty:
    # Truncate the WKT for display only — the full string is what went into the .gdb.
    wkt_column = field_name_map["coordinates_wkt"]
    preview[wkt_column] = preview[wkt_column].str.slice(0, 40) + " ..."
print(preview.to_string(index=False))

                                uuid                              coordinates_wkt        PassengerCar            Resident                Taxi           PublicBus         MediumTruck                        FRC  Net2Class         FormOfWay
00004531-3200-0400-0000-000003d07565 LINESTRING (-3.635973 40.3525425, -3.636 ... InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection                   Motorway Net2Class1   DualCarriageway
00004531-3200-0400-0000-000003d07568 LINESTRING (-3.6469681 40.3482397, -3.64 ... InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection                   Motorway Net2Class1   DualCarriageway
00004531-3200-0400-0000-000003d0756f LINESTRING (-3.6903413 40.4564488, -3.69 ... InPositiveDirection InPositiveDirection InPositiveDirection InPositiveDirection    InBothDirections             OtherMajorRoad Net2Class2   DualCarriageway
00004531-3200-0400-0000-000003d07571 LINESTRING 

In [9]:
# Value distribution per column, from the streaming counters.
for column, counter in value_counters.items():
    suffix = "+" if column in overflowed_counters else ""
    print(f"--- {column} ({len(counter)} distinct{suffix}) ---")
    for value, count in counter.most_common(15):
        print(f"  {value if value else '(NULL)':<40} {count:>12,}")
    if column in overflowed_counters:
        print(f"  (counting stopped at {counter_max_distinct} distinct values — counts are partial)")
    print()

--- PassengerCar (6 distinct) ---
  InBothDirections                               10,397
  InNegativeDirection                             8,675
  InPositiveDirection                             7,104
  InPositiveDirection|InNegativeDirection           195
  InNegativeDirection|InPositiveDirection           116
  (NULL)                                             55

--- Resident (6 distinct) ---
  InBothDirections                                9,333
  InNegativeDirection                             8,857
  InPositiveDirection                             7,224
  (NULL)                                          1,017
  InPositiveDirection|InNegativeDirection            68
  InNegativeDirection|InPositiveDirection            43

--- Taxi (6 distinct) ---
  InBothDirections                               10,336
  InNegativeDirection                             8,714
  InPositiveDirection                             7,128
  InPositiveDirection|InNegativeDirection           183
  InNegative

In [10]:
# Verify the .gdb by reading its metadata — read_info() only touches the header, so this stays
# cheap no matter how many features were written.
#
# The layer reports its geometry type as MultiLineString: a File Geodatabase polyline is a
# multi-part type by definition, so GDAL declares the layer that way even though every feature
# written here is a single-part line. Nothing is merged or lost.
info = pyogrio.read_info(gdb_output_path, layer=gdb_layer_name)

gdb_size_bytes = sum(
    os.path.getsize(os.path.join(root, name))
    for root, _, names in os.walk(gdb_output_path)
    for name in names
)

print(f"Layer          : {gdb_layer_name}")
print(f"Features stored: {info['features']:,} (features written: {rows_written:,})")
print(f"Geometry type  : {info['geometry_type']}")
print(f"CRS            : {info['crs']}")
print(f"On-disk size   : {gdb_size_bytes / 1024**3:.2f} GiB")
print(f"Ratio to input : {gdb_size_bytes / input_size_bytes:.2f}x")
print()
print("Fields as stored in the .gdb:")
for name, dtype in zip(info["fields"], info["dtypes"]):
    print(f"  {name:<32} {dtype}")

if info["features"] != rows_written:
    raise RuntimeError(
        f"Feature count mismatch: wrote {rows_written:,} but the layer holds {info['features']:,}"
    )

Layer          : dtfr
Features stored: 26,542 (features written: 26,542)
Geometry type  : MultiLineString
CRS            : EPSG:4326
On-disk size   : 0.01 GiB
Ratio to input : 0.56x

Fields as stored in the .gdb:
  uuid                             object
  coordinates_wkt                  object
  PassengerCar                     object
  Resident                         object
  Taxi                             object
  PublicBus                        object
  MediumTruck                      object
  FRC                              object
  Net2Class                        object
  FormOfWay                        object


In [11]:
# A few real features straight out of the .gdb (max_features keeps this from loading the whole
# layer). This is what ArcGIS / QGIS will show.
sample = pyogrio.read_dataframe(gdb_output_path, layer=gdb_layer_name, max_features=5)
print(f"Geometry of the first feature: {sample.geometry.iloc[0].geom_type}")
print()
if include_wkt_column:
    wkt_column = field_name_map["coordinates_wkt"]
    sample[wkt_column] = sample[wkt_column].str.slice(0, 30) + " ..."
print(sample.drop(columns="geometry").to_string(index=False))

Geometry of the first feature: MultiLineString

                                uuid                    coordinates_wkt        PassengerCar            Resident                Taxi           PublicBus         MediumTruck                        FRC  Net2Class         FormOfWay
00004531-3200-0400-0000-000003d07565 LINESTRING (-3.635973 40.35254 ... InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection                   Motorway Net2Class1   DualCarriageway
00004531-3200-0400-0000-000003d07568 LINESTRING (-3.6469681 40.3482 ... InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection InNegativeDirection                   Motorway Net2Class1   DualCarriageway
00004531-3200-0400-0000-000003d0756f LINESTRING (-3.6903413 40.4564 ... InPositiveDirection InPositiveDirection InPositiveDirection InPositiveDirection    InBothDirections             OtherMajorRoad Net2Class2   DualCarriageway
00004531-3200-0400-0000-000003d07571 LIN

In [12]:
# ---------------------------------------------------------------------------
# COORDINATE FIDELITY AUDIT — are the coordinates in the .gdb the coordinates from the JSON?
#
# This measures rather than assumes, on the first `precision_audit_features` features actually
# written. Two paths are checked independently:
#
#   * coordinates_wkt — plain text, so it should be bit-exact.
#   * the geometry     — passed through the File Geodatabase integer grid, so a deviation up to
#                        1/gdb_xy_scale is expected and inherent to the format.
#
# Features are compared positionally: FileGDB preserves insertion order via OBJECTID, and the
# audit list was filled in the same order the features were written.
# ---------------------------------------------------------------------------
audit_count = len(audit_source_coords)
audit = pyogrio.read_dataframe(gdb_output_path, layer=gdb_layer_name, max_features=audit_count)

grid_step = 1 / gdb_xy_scale
coords_checked = 0
geom_max_dev = 0.0
wkt_max_dev = 0.0
geom_beyond_grid = 0  # deviation larger than the grid can explain -> a real problem
wkt_differing = 0
vertex_count_mismatch = 0
worst_example = None


def flatten(coordinates):
    """Yield (x, y) pairs from any GeoJSON coordinate nesting depth."""
    if coordinates and isinstance(coordinates[0], (int, float)):
        yield coordinates[0], coordinates[1]
        return
    for part in coordinates or []:
        yield from flatten(part)


for position in range(audit_count):
    source = list(flatten(audit_source_coords[position]))
    geometry = audit.geometry.iloc[position]
    stored = [
        (x, y)
        for part in (geometry.geoms if hasattr(geometry, "geoms") else [geometry])
        for x, y in part.coords
    ]

    if len(source) != len(stored):
        vertex_count_mismatch += 1
        continue

    if include_wkt_column:
        body = audit[field_name_map["coordinates_wkt"]].iloc[position].split("(", 1)[1]
        wkt_points = [
            tuple(float(v) for v in point.split()[:2])
            for point in body.replace(")", "").split(",")
        ]
    else:
        wkt_points = stored  # nothing to check; compare against itself

    for (sx, sy), (gx, gy), (wx, wy) in zip(source, stored, wkt_points):
        coords_checked += 1

        geom_dev = max(abs(sx - gx), abs(sy - gy))
        if geom_dev > geom_max_dev:
            geom_max_dev = geom_dev
            worst_example = ((sx, sy), (gx, gy))
        if geom_dev > grid_step:
            geom_beyond_grid += 1

        if include_wkt_column:
            wkt_dev = max(abs(sx - wx), abs(sy - wy))
            wkt_max_dev = max(wkt_max_dev, wkt_dev)
            if wkt_dev:
                wkt_differing += 1

print(f"Features audited        : {audit_count:,} (of {rows_written:,} written)")
print(f"Coordinates compared    : {coords_checked:,}")
print(f"Vertex-count mismatches : {vertex_count_mismatch:,}")
print()
print(f"XY grid step            : {grid_step:.1e} deg (XYSCALE={gdb_xy_scale:.0e})")
print(f"GEOMETRY max deviation  : {geom_max_dev:.3e} deg (~{geom_max_dev * 111_320_000:.3g} mm)")
print(f"  beyond the grid step  : {geom_beyond_grid:,}")
print(f"  tolerance             : {max_geometry_deviation_mm:g} mm")
if include_wkt_column:
    print(f"WKT text max deviation  : {wkt_max_dev:.3e} deg  ({wkt_differing:,} coordinates differ)")
if worst_example:
    print(f"  largest geometry diff : source={worst_example[0]} stored={worst_example[1]}")
print()

problems = []
if vertex_count_mismatch:
    problems.append(f"{vertex_count_mismatch:,} features changed vertex count")
if geom_beyond_grid:
    problems.append(
        f"{geom_beyond_grid:,} coordinates moved further than the {grid_step:.1e} deg grid explains"
    )
if include_wkt_column and wkt_differing:
    problems.append(f"{wkt_differing:,} WKT coordinates differ from the source text")
if geom_max_dev * 111_320_000 > max_geometry_deviation_mm:
    problems.append(
        f"geometry moved up to {geom_max_dev * 111_320_000:.3g} mm, beyond the "
        f"{max_geometry_deviation_mm:g} mm tolerance — the XY grid is too coarse for this source"
    )

if problems:
    raise RuntimeError(
        "Coordinate fidelity check FAILED: "
        + "; ".join(problems)
        + ". Raise gdb_xy_scale (currently "
        f"{gdb_xy_scale:.0e}) if the source carries more decimals than the grid can hold."
    )

print(
    "PASS — no coordinate was truncated beyond the geodatabase's own grid."
    + (
        "\n  The geometry deviation above is the grid round-trip (round(coord * XYSCALE) / XYSCALE)"
        "\n  and is inherent to the .gdb format; coordinates_wkt holds the exact source text."
        if include_wkt_column
        else ""
    )
)

Features audited        : 2,000 (of 26,542 written)
Coordinates compared    : 7,991
Vertex-count mismatches : 0

XY grid step            : 1.0e-12 deg (XYSCALE=1e+12)
GEOMETRY max deviation  : 2.842e-14 deg (~3.16e-06 mm)
  beyond the grid step  : 0
  tolerance             : 1 mm
WKT text max deviation  : 0.000e+00 deg  (0 coordinates differ)
  largest geometry diff : source=(-3.635973, 40.3525425) stored=(-3.6359729999999786, 40.35254250000003)

PASS — no coordinate was truncated beyond the geodatabase's own grid.
  The geometry deviation above is the grid round-trip (round(coord * XYSCALE) / XYSCALE)
  and is inherent to the .gdb format; coordinates_wkt holds the exact source text.
